In [ ]:
#| default_exp features.fsc_gene

In [ ]:
#| export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()


In [ ]:
#| export
class FSCGeneEvaluator(FeatureEvaluator):
    """Extracts gene-level fragment size characteristics."""
    
    name = "FSC_gene"
    source_file = ".FSC.gene.parquet"
    tier = 1
    category = "fragmentation"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        """Transform the loaded raw dataframe into meaningful scalar metrics per sample."""
        extracted = {}
        try:
            if df.empty:
                return extracted
                
            # Need to safeguard columns if they differ
            cols = set(df.columns)
            
            # Global aggregates across all genes
            if "core_short_ratio" in cols:
                extracted["global_core_short_ratio_mean"] = df["core_short_ratio"].mean()
            if "mono_nucl_ratio" in cols:
                extracted["global_mono_nucl_ratio_mean"] = df["mono_nucl_ratio"].mean()
                
            if "normalized_depth" in cols:
                m_depth = df["normalized_depth"].mean()
                if pd.notna(m_depth) and m_depth > 0:
                    extracted["global_normalized_depth_cv"] = df["normalized_depth"].std() / m_depth
                else:
                    extracted["global_normalized_depth_cv"] = 0.0

            # Per-gene extraction (128 genes)
            if "gene" in cols:
                for _, row in df.iterrows():
                    g = str(row["gene"]).upper().replace(" ", "_").replace("-", "_")
                    if g == "NAN":
                        continue
                    
                    if "core_short_ratio" in cols:
                        extracted[f"{g}_core_short_ratio"] = float(row["core_short_ratio"])
                    if "mono_nucl_ratio" in cols:
                        extracted[f"{g}_mono_nucl_ratio"] = float(row["mono_nucl_ratio"])
                    if "normalized_depth" in cols:
                        extracted[f"{g}_normalized_depth"] = float(row["normalized_depth"])
                        
            return extracted
        except Exception as e:
            log.warning("fsc_gene_extraction_failed", error=str(e))
            return {}


In [ ]:
#| test
evaluator = FSCGeneEvaluator()
assert evaluator.name == "FSC_gene"
assert evaluator.tier == 1

# Dummy dataframe to test extraction safety
df_test = pd.DataFrame([{
    "gene": "TP53",
    "core_short_ratio": 0.15,
    "mono_nucl_ratio": 0.50,
    "normalized_depth": 1.2
}])

res = evaluator.extract(df_test)
assert "global_core_short_ratio_mean" in res
assert "TP53_core_short_ratio" in res
assert res["TP53_core_short_ratio"] == 0.15
